# Evaluating the Warehouse Agent with Strands Evals & AgentCore Evaluations

In Lab 7 you deployed a warehouse operations agent to Amazon Bedrock AgentCore Runtime. But an answer that *looks* right is not proof the agent is good. Before you trust it with real procurement decisions, you need to measure its quality in a repeatable way.

This lab builds an evaluation suite and grades the agent in **two stages**. Stage 1 runs the agent locally, in this notebook, with [Strands Evals](https://pypi.org/project/strands-agents-evals/) for fast, cheap feedback while you iterate. Stage 2 grades the agent on AgentCore Runtime with [AgentCore Evaluations](https://docs.aws.amazon.com/bedrock-agentcore/), reading the production traces the deployed agent emits. The grader design follows Anthropic's [Demystifying evals for AI agents](https://www.anthropic.com/engineering/demystifying-evals-for-ai-agents).

> **This notebook runs on its own.** The setup section sets up everything it needs (the model, the local agent, and the deployment details), so no other lab needs to be loaded in the same kernel. Stage 1 needs only your SAP GenAI Hub credentials. Stage 2 also needs the runtime you deployed in Lab 7.

If you have not set up your AI Core credentials yet, follow notebook 00-load-sap-ai-core-credentials first.


## Prerequisites

1. **Completed Lab 7**, which deploys a warehouse agent to AgentCore Runtime and writes `lab7_deployment.json`. Required for Stage 2.
2. SAP AI Core credentials in `~/.aicore/config.json` (run Lab 00 if you have not).
3. An SAP S/4HANA Public Cloud API key, either in `.env` or entered when prompted.
4. AWS credentials with AgentCore Evaluation permissions.
5. **CloudWatch Transaction Search enabled** in this account/region (required for Stage 2). AgentCore Runtime emits OpenTelemetry spans automatically — the starter toolkit builds the container with `aws-opentelemetry-distro` and runs it under `opentelemetry-instrument`, so you do **not** configure spans in the agent code. But those spans only become queryable traces once **Transaction Search** is turned on at the account level; without it, `EvaluationClient.run()` finds no trace to grade. The Stage 2 preflight cell checks this for you and prints setup instructions if it is off. See [Transaction Search setup](https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/observability-configure.html).

Stage 1 needs only items 2 and 3. Stage 2 also needs the deployed runtime (item 1), AWS permissions (item 4), and Transaction Search (item 5).

## 1. Setup: imports, configuration, and the local agent

The next few cells import dependencies, check your credentials, initialize the `SAPGenAIHubModel`, load the deployed-agent details from `lab7_deployment.json`, and build the local warehouse agent used in Stage 1.


In [ ]:
from util.strands_bedrock_sap_genai_hub import SAPGenAIHubModel

import os
import json
import time
import uuid
from datetime import timedelta
from collections import defaultdict

import boto3
from dotenv import load_dotenv
import getpass

load_dotenv()

# Prompt for the SAP API key if it isn't already set
if not os.environ.get("SAP_S4HANA_PUBLIC_CLOUD_KEY"):
    os.environ["SAP_S4HANA_PUBLIC_CLOUD_KEY"] = getpass.getpass("SAP_S4HANA_PUBLIC_CLOUD_KEY:\n")


In [ ]:
# Validate required configuration before proceeding
_errors = []

if not os.path.exists(os.path.expanduser("~/.aicore/config.json")):
    _errors.append("Missing ~/.aicore/config.json. Run notebook 00 first.")

if not os.environ.get("SAP_S4HANA_PUBLIC_CLOUD_KEY"):
    _errors.append("SAP_S4HANA_PUBLIC_CLOUD_KEY not set. Check your .env file.")

try:
    _sts = boto3.client("sts").get_caller_identity()
    print(f"AWS Identity: {_sts['Arn']}")
except Exception as e:
    _errors.append(f"AWS credentials not configured: {e}")

if _errors:
    for err in _errors:
        print(f"ERROR: {err}")
    raise SystemExit("Fix the errors above before continuing.")
else:
    print("All prerequisites validated.")

In [ ]:
# TODO: Choose your model. Options: "anthropic--claude-4.5-sonnet", "amazon--nova-lite", "amazon--nova-pro"
model = SAPGenAIHubModel(
    model_id="anthropic--claude-4.5-sonnet",
    max_tokens=4096,
)

# Load the deployed-agent details written by Lab 7 (lab7_deployment.json). Stage 2 evaluates
# this deployed runtime. Stage 1 only needs `model`, but we load the deployment up front so
# the whole lab is configured in one place.
DEPLOYMENT_FILE = "lab7_deployment.json"
if not os.path.exists(DEPLOYMENT_FILE):
    raise SystemExit(
        f"{DEPLOYMENT_FILE} not found. Run Lab 7 "
        "(07-deploy-warehouse-agent-to-agentcore.ipynb) first."
    )

with open(DEPLOYMENT_FILE, "r") as f:
    deployment = json.load(f)

AGENT_NAME = deployment["agent_name"]
AGENT_ID = deployment["agent_id"]
AGENT_ARN = deployment["agent_arn"]
REGION = deployment.get("region") or "us-east-1"

print(f"Region: {REGION}")
print(f"Agent Name: {AGENT_NAME}")
print(f"Agent ID: {AGENT_ID}")
print(f"Agent ARN: {AGENT_ARN}")
print(f"Model: {model.get_config()['model_id']} via SAP GenAI Hub")


### The local agent

Stage 1 grades a local copy of the same warehouse agent you built in Lab 6 and deployed in Lab 7. It is a two-agent design: a **selector sub-agent** (`SelectorAPIAgentAsATool`) reads the OpenAPI specs under `assets/knowledgebase` and picks which SAP API fits the question, then the **warehouse agent** calls that selector and the `odata_caller` tool to fetch inventory data and answer.

Running it here, in process, lets us read its tool calls straight off the result. One thing to know: **the baseline is deliberately naive.** Its prompt tells the agent to discover the schema at runtime (call the selector, then read `$metadata`, then hand-build the OData query). That is flexible but wasteful, and it is exactly the inefficiency Stage 1's trajectory judge catches. Later in Stage 1 we swap in an **improved agent** whose architecture fixes what the judge names, and re-measure.

Both agents live in `util/warehouse_agent.py` (`build_warehouse_agent` and `build_improved_warehouse_agent`), extracted from Lab 6 so this notebook stays focused on evaluation.


In [ ]:
# Import the two Lab 6/7 warehouse-agent architectures from util/warehouse_agent.py:
#  - build_warehouse_agent: the BASELINE (selector sub-agent + odata_caller). It rediscovers the
#    schema at runtime via the selector + $metadata, and hand-builds $filter/$select. Stage 1
#    grades this as the naive baseline.
#  - build_improved_warehouse_agent: the eval-driven FIX. A deep `get_warehouse_stock` tool encodes
#    the entity, fields, warehouse ID, and server-side query, so a stock question is answered in one
#    call. The selector + odata_caller stay wired in as a fallback for off-path queries (a router).
# See util/warehouse_agent.py for the finding-by-finding mapping from the trajectory judge to the fix.
from util.warehouse_agent import (
    WAREHOUSE_SYSTEM_PROMPT,
    WAREHOUSE_CAPACITY,
    build_warehouse_agent,
    build_improved_warehouse_agent,
)


def create_warehouse_agent():
    """Build the local BASELINE warehouse agent (selector + odata_caller) for Stage 1."""
    return build_warehouse_agent(model, system_prompt=WAREHOUSE_SYSTEM_PROMPT)


def create_improved_warehouse_agent():
    """Build the local IMPROVED warehouse agent (deep get_warehouse_stock tool + fallback)."""
    return build_improved_warehouse_agent(model)


# Smoke-test that both agents construct.
_ = create_warehouse_agent()
_ = create_improved_warehouse_agent()
print("Local warehouse agents (baseline + improved) ready for Stage 1 evaluation.")


## How we evaluate: two stages

The two stages answer two different questions:

1. **Stage 1, local, with [Strands Evals](https://pypi.org/project/strands-agents-evals/): "is the logic good?"** Run the agent in this notebook and grade it in process. Fast, cheap, and no deployment needed. This is the pre-deploy gate you iterate and *fix* against before you ship.
2. **Stage 2, deployed, with [AgentCore Evaluations](https://docs.aws.amazon.com/bedrock-agentcore/): "did the fix ship, and how does the real thing behave under production infrastructure?"** We ship Stage 1's fix to the live AgentCore Runtime, then grade it from the production OpenTelemetry (OTEL) traces it writes to CloudWatch. That means a **parity** check (did quality hold?) plus **prod-only signals** a local run cannot produce: latency and cold-start. This is also what you wire into ongoing production monitoring.

Grade locally as you build, then ship the fix and verify it in production.

```
  STAGE 1: Local (this notebook)                 STAGE 2: Deployed (AgentCore Runtime)
  +-------------------------------+             +--------------------------------------------+
  | baseline vs. improved agent   |             | redeploy improved agent (deep tool)         |
  |        | run scenario         |             |        | invoke_agent_runtime()  OTEL  CW    |
  |        v                      |             |        v   EvaluationClient.run()           |
  | Strands Evals                 |             | AgentCore built-in + custom evaluators      |
  |  - ToolCalled (code)          |             |  - Correctness / Helpfulness (LLM-judge)    |
  |  - OutputEvaluator (LLM)      |             |  - GoalSuccessRate / ToolSelectionAccuracy  |
  |  - TrajectoryEvaluator (LLM)  |             |  - WarehouseOperationalQuality (custom)     |
  |  - ToolSelectionAccuracy (LLM)|             |  + prod-only: latency, cold-start           |
  +-------------------------------+             +--------------------------------------------+
     is the logic good? (pre-deploy gate)          did the fix ship? how does prod behave?
```


## Understanding grader types

Anthropic's [Demystifying evals for AI agents](https://www.anthropic.com/engineering/demystifying-evals-for-ai-agents) groups graders into three families. Most real evaluations **combine** them.

| Grader family | How it works | In this lab |
|---|---|---|
| **Code-based** | Deterministic checks: string match, counting, static analysis | `ToolCalled` (local) |
| **Model-based (LLM as judge)** | An LLM scores the output against a rubric | `OutputEvaluator`, `TrajectoryEvaluator`, `ToolSelectionAccuracy` (local); `Correctness`, `Helpfulness`, `GoalSuccessRate`, `ToolSelectionAccuracy`, and the custom `WarehouseOperationalQuality` (deployed) |
| **Human** | Expert review and spot checks | Not shown here; used in practice to calibrate the LLM judges |

Rule of thumb: use code-based graders wherever you can (cheapest and most reliable), reach for LLM judges when you need nuance, and use humans to calibrate those judges.

**There is a second question to ask about every grader: *what* is it grading?**

- **Outcome** grades the final answer. *Did the agent get it right?* Examples: `OutputEvaluator` (local), `Correctness` and `WarehouseOperationalQuality` (deployed).
- **Trajectory** grades the path taken. *Did it use the right tools, efficiently?* Examples: `ToolCalled`, `TrajectoryEvaluator`, `ToolSelectionAccuracy`.

You want both. Our `low-stock-items` scenario shows why: the agent reaches the correct answer (outcome passes) but takes a wasteful path of redundant OData calls (trajectory fails). An outcome-only score would call that a clean pass and hide the problem.


## Evaluation scenarios

These scenarios are **shared by both stages**: graded locally in Stage 1 and on the deployed runtime in Stage 2. Each scenario carries four things:

- **prompt**: the user query the agent must answer.
- **expected_response**: what a good answer looks like, used by the outcome graders (`OutputEvaluator` locally, `Correctness` and the custom evaluator when deployed).
- **expected_trajectory**: the tool calls we expect, used by the trajectory graders (`ToolSelectionAccuracy`, `TrajectoryEvaluator`).
- **assertions**: the goal conditions, used by `GoalSuccessRate` in Stage 2.

The last scenario, `low-stock-items`, is a real workshop failure. The query triggers many redundant OData calls (repeated `$metadata` reads, trial-and-error `$filter` guessing) yet still returns a correct answer. Outcome graders score it 1.0, so the inefficiency is invisible to them. The `TrajectoryEvaluator` in Stage 1 is what catches the wasteful path and explains it.


In [ ]:
evaluation_scenarios = [
    {
        "name": "inventory-check-single",
        "prompt": "What is the current stock level for WM-AN02 Control Units?",
        "expected_response": (
            "The response should contain the specific stock quantity for WM-AN02 Control Units "
            "retrieved from the SAP warehouse API, with the product correctly identified as "
            "Control Units. The number should come from actual API data, not be invented."
        ),
        "expected_trajectory": ["odata_caller"],
        "assertions": (
            "Agent queried warehouse stock data via the OData API. "
            "Agent correctly identified WM-AN02 as Control Units. "
            "Agent reported a specific numeric stock quantity from the API."
        ),
    },
    {
        "name": "inventory-overview",
        "prompt": "Give me a complete overview of all products currently in the warehouse.",
        "expected_response": (
            "The response should list all warehouse products (WM-AN01 Advanced Sensors, "
            "WM-AN02 Control Units, WM-AN03 Power Modules, WM-AN04 Communication Devices) "
            "with their current stock quantities from the API, presented in a structured format."
        ),
        "expected_trajectory": ["odata_caller"],
        "assertions": (
            "Agent queried warehouse stock data via OData. "
            "Agent listed multiple products with stock quantities. "
            "Agent presented results in a structured, readable format."
        ),
    },
    {
        "name": "fulfillment-feasibility",
        "prompt": "Can we fulfill an order for 200 units of WM-AN02 Control Units?",
        "expected_response": (
            "The response should check current WM-AN02 stock from the API, compare it against "
            "the requested 200 units, and provide a clear yes/no fulfillment recommendation "
            "with the actual available quantity."
        ),
        "expected_trajectory": ["odata_caller"],
        "assertions": (
            "Agent queried current WM-AN02 stock via OData. "
            "Agent compared available quantity against the 200-unit request. "
            "Agent provided a clear yes/no fulfillment answer with supporting data."
        ),
    },
    {
        # Regression scenario from a real workshop failure. This query used to trigger many
        # redundant OData calls (repeated $metadata discovery, trial-and-error $filter guessing).
        # Correctness/GoalSuccessRate scored it 1.0 despite the waste, so we lean on the
        # LLM-as-judge trajectory grader (Stage 1) to catch and explain the inefficiency.
        "name": "low-stock-items",
        "prompt": "What are the items with low stock?",
        "expected_response": (
            "The response should identify which products are low on stock (or state that none "
            "are below the reorder threshold), based on stock quantities from the SAP warehouse "
            "API. The agent should reach the answer efficiently, ideally a single filtered or "
            "sorted OData query, rather than fetching everything and repeatedly rediscovering the schema."
        ),
        "expected_trajectory": ["odata_caller"],
        "assertions": (
            "Agent queried warehouse stock data via the OData API. "
            "Agent determined which products are low on stock relative to their capacity. "
            "Agent reached the answer without redundant, repeated OData calls."
        ),
    },
]

print(f"Evaluation scenarios defined: {len(evaluation_scenarios)}")
for s in evaluation_scenarios:
    print(f"  - {s['name']}: {s['prompt']}")

## Stage 1: Local evaluation with Strands Evals

Run the agent right here in the notebook and grade it in process. This is the fast, cheap gate you iterate against before deploying. **[Strands Evals](https://pypi.org/project/strands-agents-evals/)** (already a project dependency) gives us graders from both families:

- **Code-based (deterministic):** `ToolCalled` checks that the agent used its tool at all.
- **LLM as judge:** `OutputEvaluator` scores whether the final answer is correct (task success), `TrajectoryEvaluator` scores the tool-call path against a rubric, and `ToolSelectionAccuracyEvaluator` judges whether each individual call was justified. All three route through the **same `SAPGenAIHubModel`** the agent uses, so no separate Bedrock access is needed.

**Where the trajectory comes from.** Each scenario runs once against the local agent. We read its tool usage off the result with the SDK's `tools_use_extractor` ([docs](https://strandsagents.com/docs/user-guide/evals-sdk/quickstart/)):

```python
from strands_evals.extractors import tools_use_extractor
calls = tools_use_extractor.extract_agent_tools_used_from_messages(agent.messages)
# -> [{"name": "odata_caller", "input": {...}, "tool_result": "...", "is_error": False}, ...]
```

We keep two views of that run: a flat list of tool names for `ToolCalled`, and a detailed list (each call's OData arguments plus an `is_error` flag) for the trajectory judge, so it can name the *specific* redundant or failed calls instead of just counting them. This is what lets a scenario **pass on task success but score low on trajectory**: the right answer reached the wrong way. That is the `low-stock-items` case, and it is exactly what an outcome-only score misses.


In [ ]:
# Code-based grader with Strands Evals: deterministic, no LLM. This cell also defines the
# reusable `evaluate_agent` helper the "fix and re-test" cell below calls a second time.
from strands_evals.evaluators import ToolCalled
from strands_evals.extractors import tools_use_extractor
from strands_evals.types import EvaluationData

# The BASELINE agent answers via the generic odata_caller; the IMPROVED agent answers via the deep
# get_warehouse_stock tool. evaluate_agent takes the primary tool name so it counts and grades the
# right one for whichever architecture is under test (the whole before/after point is that this
# tool changes). TODO: change if your agent uses a different primary tool.
TOOL_NAME = "odata_caller"
IMPROVED_TOOL_NAME = "get_warehouse_stock"

# OData argument keys worth surfacing to the trajectory judge: these reveal waste
# (repeated $metadata rediscovery, trial-and-error $filter guessing).
_ODATA_KEYS_OF_INTEREST = ("endpoint", "operation", "$filter", "$orderby", "$select", "$top")

# Captured once so the trajectory judge knows what each tool does (see the LLM-judge cell).
TOOL_DESCRIPTIONS = {}


def _summarize_tool_call(call: dict) -> dict:
    """Reduce one extracted tool-call record to the fields that show what it did.

    We surface the endpoint and OData query params (so the judge sees $filter/$orderby
    directly) and keep is_error, since a failed call is the tell-tale of trial-and-error
    $filter guessing. For the deep get_warehouse_stock tool we surface its typed args instead,
    so the judge can see the whole query was expressed in a single well-formed call.
    """
    tool_input = call.get("input") or {}
    odata_params = tool_input.get("odata_params") or {}
    flat = {**tool_input, **odata_params}
    summary = {"name": call.get("name")}
    for key in (*_ODATA_KEYS_OF_INTEREST, "product", "low_stock_only"):
        if flat.get(key) is not None and flat.get(key) != "":
            summary[key] = flat[key]
    if call.get("is_error"):
        summary["is_error"] = True
    return summary


def run_local_agent_trajectory(prompt: str, make_agent):
    """Run one scenario against a local agent; return (output, names, detailed, tokens).

    - names: flat tool-name list, for `ToolCalled` (exact-match membership).
    - detailed: ordered {name, endpoint, $filter, is_error, ...} dicts, for the LLM judge.
    - tokens: total tokens the agent consumed (informational efficiency signal).
    Both trajectory views come from the SDK's `tools_use_extractor`.
    """
    global TOOL_DESCRIPTIONS
    agent = make_agent()
    result = agent(prompt)
    output_text = str(result.message)

    if not TOOL_DESCRIPTIONS:
        TOOL_DESCRIPTIONS = tools_use_extractor.extract_tools_description(agent)

    calls = tools_use_extractor.extract_agent_tools_used_from_messages(agent.messages)
    names = [c.get("name") for c in calls]
    detailed = [_summarize_tool_call(c) for c in calls]
    tokens = (result.metrics.accumulated_usage or {}).get("totalTokens")
    return output_text, names, detailed, tokens


def evaluate_agent(make_agent, scenarios, tool_name=TOOL_NAME):
    """Run every scenario against `make_agent()` once and grade with the code-based `ToolCalled`.

    `tool_name` is the primary data tool for the architecture under test (odata_caller for the
    baseline, get_warehouse_stock for the improved agent); ToolCalled checks it was used and
    n_tool_calls counts it. The expected trajectory is set to that tool too, so the trajectory
    graders compare against the right target. Returns a per-scenario cache reused by the LLM-judge
    cell and the before/after comparison, so we invoke the agent once per scenario, not per grader.
    """
    runs = {}
    for scenario in scenarios:
        name = scenario["name"]
        output_text, names, detailed, tokens = run_local_agent_trajectory(scenario["prompt"], make_agent)
        expected_trajectory = [tool_name]

        # name_case carries expected_output so the OutputEvaluator (task-success judge) can
        # compare the answer against what a good response should contain.
        name_case = EvaluationData(
            name=name, input=scenario["prompt"], actual_output=output_text,
            expected_output=scenario["expected_response"],
            actual_trajectory=names, expected_trajectory=expected_trajectory,
        )
        detail_case = EvaluationData(
            name=name, input=scenario["prompt"], actual_output=output_text,
            actual_trajectory=detailed, expected_trajectory=expected_trajectory,
        )

        tool_called = ToolCalled(tool_name).evaluate(name_case)[0]
        n_calls = names.count(tool_name)
        runs[name] = {
            "name_case": name_case,
            "detail_case": detail_case,
            "names": names,
            "detailed": detailed,
            "output": output_text,
            "tokens": tokens,
            "tool_called_result": {
                "evaluatorId": "StrandsEvals.ToolCalled",
                "value": tool_called.score,
                "label": "called" if tool_called.test_pass else "not_called",
                "explanation": tool_called.reason,
                "n_tool_calls": n_calls,
                "tool_called": tool_called.test_pass,
            },
        }
        tok = f"{tokens} tokens" if tokens is not None else "tokens n/a"
        print(f"  {name}: tool_called={tool_called.test_pass} ({n_calls} '{tool_name}' call(s), {tok})")
    return runs


print("Running Strands Evals code-based grader on the baseline agent...\n")
local_runs = evaluate_agent(create_warehouse_agent, evaluation_scenarios)
tool_called_results = {n: [r["tool_called_result"]] for n, r in local_runs.items()}
print("\nCode-based grading complete.")

### LLM-as-judge graders (local)

`ToolCalled` tells you the agent reached its tool, not whether it answered well or worked efficiently. For that we add three LLM-as-judge graders, all routed through the same `SAPGenAIHubModel`:

- **`OutputEvaluator`** (the outcome judge, [docs](https://strandsagents.com/docs/user-guide/evals-sdk/evaluators/output_evaluator/)) reads the agent's final answer and the scenario's `expected_response` and scores **task success** on content alone: did the agent return the right inventory data, with the right product codes, without inventing numbers?
- **`TrajectoryEvaluator`** scores the tool-call path against a rubric. Fed the detailed trajectory (endpoints, OData arguments, `is_error` flags), it names the specific redundant or failed calls, so its reason is a diagnosis you can act on.
- **`ToolSelectionAccuracyEvaluator`** is a call-level judge ("was this call justified?"). It needs a Strands `Session` built from OTEL spans, so we drive it through the `Experiment` and `TracedHandler` harness, which captures spans from a fresh local run.

Together the outcome judge and the trajectory judge answer two different questions about the same run: **was the answer right, and was the path to it efficient?**

> These graders make **live LLM calls**, so this cell is slower than the deterministic grader and scores can vary slightly between runs. The `ToolSelectionAccuracyEvaluator` path re-runs the agent to capture traces; if local span capture is unavailable it is skipped and the `OutputEvaluator` and `TrajectoryEvaluator` results still stand.


In [ ]:
# LLM-as-judge graders with Strands Evals, routed through the same SAPGenAIHubModel.
from strands_evals.evaluators import (
    OutputEvaluator,
    TrajectoryEvaluator,
    ToolSelectionAccuracyEvaluator,
)

# Rubric for the OUTPUT judge: reads the final answer and the scenario's expected_response and
# scores task success on content alone, ignoring the path taken.
OUTPUT_RUBRIC = (
    "You are grading a warehouse inventory agent's final answer for TASK SUCCESS. Compare the "
    "agent's output against the expected response, judging factual content only, ignore wording, "
    "formatting, and style. A successful answer contains the specific inventory data the question "
    "asked for (stock quantities, correct product codes such as WM-AN02, a clear yes/no where a "
    "decision was requested) and does not invent numbers.\n\n"
    "Score 1.0 if the answer fully satisfies the expected response, 0.5 if it is partially correct "
    "or missing key data, and 0.0 if it is wrong, empty, or hallucinated. State briefly in your "
    "reason which required facts are present or missing."
)

# Rubric for the TRAJECTORY judge. We ask the judge to NAME the specific redundant calls it sees,
# so its `reason` becomes an actionable diagnosis rather than just a score.
TRAJECTORY_RUBRIC = (
    "Score the tool-call trajectory for a warehouse inventory agent. Each entry shows the tool "
    "name and the OData arguments used ($filter, $orderby, endpoint, etc.); entries with "
    "`is_error: true` are calls that FAILED. A good trajectory answers the user's question with "
    "as few tool calls as possible, ideally one filtered or sorted OData query.\n\n"
    "Penalize redundant work and, in your reason, IDENTIFY THE SPECIFIC WASTEFUL CALLS: "
    "repeated `$metadata` schema rediscovery, trial-and-error `$filter` guessing (a failed call "
    "followed by retries that differ only in a fumbled filter, the `is_error` flags mark these), "
    "or fetching everything and filtering client-side when a server-side query would do.\n\n"
    "Score 1.0 for an efficient, well-chosen trajectory; lower toward 0.0 as redundant or failed "
    "calls increase. Your reason must explain WHICH calls were wasteful and WHY, and name the "
    "concrete fix (e.g. 'the second and third calls re-fetched $metadata; put the field names in "
    "the system prompt so the agent filters on the first call')."
)

llm_judge_results = {}

print("Running Strands Evals LLM-as-judge graders (via SAP GenAI Hub)...\n")

# OutputEvaluator: the task-success (outcome) judge. include_inputs=True gives the judge the
# original question for context.
output_judge = OutputEvaluator(rubric=OUTPUT_RUBRIC, model=model, include_inputs=True)

for scenario_name, run in local_runs.items():
    try:
        out = output_judge.evaluate(run["name_case"])[0]
        llm_judge_results[scenario_name] = [{
            "evaluatorId": "StrandsEvals.OutputEvaluator",
            "value": out.score,
            "label": out.label or ("pass" if out.test_pass else "fail"),
            "explanation": out.reason,
        }]
        print(f"  {scenario_name}: OutputEvaluator={out.score:.2f} ({'success' if out.test_pass else 'fail'})")
    except Exception as e:
        print(f"  {scenario_name}: OutputEvaluator ERROR: {e}")
        llm_judge_results[scenario_name] = []

print()

# TrajectoryEvaluator: fed the DETAILED trajectory (call args), so it can name the waste.
# `trajectory_description` tells the judge what each tool does (from extract_tools_description).
trajectory_judge = TrajectoryEvaluator(
    rubric=TRAJECTORY_RUBRIC,
    model=model,
    trajectory_description=TOOL_DESCRIPTIONS or None,
)

for scenario_name, run in local_runs.items():
    try:
        out = trajectory_judge.evaluate(run["detail_case"])[0]
        llm_judge_results.setdefault(scenario_name, []).append({
            "evaluatorId": "StrandsEvals.TrajectoryEvaluator",
            "value": out.score,
            "label": out.label or ("pass" if out.test_pass else "fail"),
            "explanation": out.reason,
        })
        # Print the full reason: this is the "why is it inefficient" diagnosis we want to read.
        print(f"  {scenario_name}: TrajectoryEvaluator={out.score:.2f}")
        print(f"    {out.reason}\n")
    except Exception as e:
        print(f"  {scenario_name}: TrajectoryEvaluator ERROR: {e}\n")

# ToolSelectionAccuracyEvaluator: tool-level judge that needs a Session (OTEL spans). We use the
# Strands Evals Experiment + TracedHandler harness, which runs the agent and captures spans into
# a Session the judge can parse. This is heavier (re-runs the agent) and depends on local
# telemetry capture, so we guard it and fall back gracefully.
#
# We `await` run_evaluations_async rather than the sync run_evaluations: the sync wrapper calls
# asyncio.run() internally, which raises inside a Jupyter kernel (it already has a running event
# loop). max_workers=1 keeps runs sequential (one SAP GenAI Hub call at a time).
try:
    from strands_evals import Experiment, Case, eval_task, TracedHandler

    cases = [
        Case(
            name=s["name"],
            input=s["prompt"],
            expected_trajectory=s["expected_trajectory"],
        )
        for s in evaluation_scenarios
    ]

    @eval_task(TracedHandler())
    def warehouse_eval_task():
        return create_warehouse_agent()

    experiment = Experiment(
        cases=cases,
        evaluators=[ToolSelectionAccuracyEvaluator(model=model)],
    )
    reports = await experiment.run_evaluations_async(warehouse_eval_task, max_workers=1)

    # Each EvaluationReport carries parallel lists: cases[i] (a dict) and scores[i].
    # Fold the tool-selection scores into llm_judge_results, keyed by scenario name.
    for report in reports:
        for case_dict, score in zip(report.cases, report.scores):
            name = case_dict.get("name") if isinstance(case_dict, dict) else None
            if name is not None and score is not None:
                llm_judge_results.setdefault(name, []).append({
                    "evaluatorId": "StrandsEvals.ToolSelectionAccuracy",
                    "value": score,
                    "label": "justified" if score >= 0.5 else "unjustified",
                    "explanation": f"{report.evaluator_name} tool-selection score.",
                })
                print(f"  {name}: ToolSelectionAccuracy={score:.2f}")
    print("\nLLM-as-judge grading complete.")
except Exception as e:
    print(f"\n  ToolSelectionAccuracyEvaluator skipped (local trace capture unavailable): {e}")
    print("  OutputEvaluator and TrajectoryEvaluator results above still stand.")

### Local results

We combine the code-based and LLM-judge scores (plus token counts) into one table. The signal to
look for: a scenario that **reached an answer but scored low on the trajectory judge**, correct
but inefficient, exactly what an outcome-only grader misses. For any flagged scenario we print the
judge's explanation and the detailed trajectory so you can see the redundant calls.


In [ ]:
# Combine local Strands Evals scores (code-based + LLM-judge) into one table.
local_results = {}
for scenario in evaluation_scenarios:
    name = scenario["name"]
    local_results[name] = (
        tool_called_results.get(name, [])
        + llm_judge_results.get(name, [])
    )

LOCAL_EVALUATOR_IDS = [
    "StrandsEvals.ToolCalled",
    "StrandsEvals.OutputEvaluator",
    "StrandsEvals.TrajectoryEvaluator",
    "StrandsEvals.ToolSelectionAccuracy",
]

# TrajJudge score below this counts as an inefficient trajectory for the flagging logic.
TRAJECTORY_PASS_THRESHOLD = 0.5


def _local_short_name(eid):
    return {
        "StrandsEvals.ToolCalled": "ToolCalled",
        "StrandsEvals.OutputEvaluator": "TaskSuccess",
        "StrandsEvals.TrajectoryEvaluator": "TrajJudge",
        "StrandsEvals.ToolSelectionAccuracy": "ToolSelect",
    }.get(eid, eid.split(".")[-1][:12])


print("=" * 100)
print(" LOCAL EVALUATION - STRANDS EVALS (code-based + LLM-judge)")
print("=" * 100)

# Tokens is an informational efficiency signal (not a pass/fail grade): fewer tokens for the
# same correct answer is a cheaper, tighter trajectory.
header = f"{'Scenario':<25}"
for eid in LOCAL_EVALUATOR_IDS:
    header += f" {_local_short_name(eid):>14}"
header += f" {'Tokens':>10}"
print(header)
print("-" * 100)

local_scenario_scores = defaultdict(dict)
for name, results in local_results.items():
    row = f"{name:<25}"
    for eid in LOCAL_EVALUATOR_IDS:
        score = next((r.get("value", "-") for r in results if r.get("evaluatorId") == eid), "-")
        if isinstance(score, (int, float)):
            row += f" {score:>14.2f}"
            local_scenario_scores[name][eid] = score
        else:
            row += f" {str(score):>14}"
    tokens = local_runs.get(name, {}).get("tokens")
    row += f" {tokens:>10}" if isinstance(tokens, int) else f" {'-':>10}"
    print(row)
print("=" * 100)

# Highlight correct-but-inefficient: the task succeeded (TaskSuccess high) but the LLM trajectory
# judge scored the path low. The judge's reason names why it's wasteful, and we print the detailed
# trajectory beside it so you can see the redundant calls. This is the gap an outcome-only score
# misses: a right answer reached the wrong way.
print("\nCorrect-but-inefficient check (task succeeded, but low trajectory-judge score):")
flagged = False
for name, results in local_results.items():
    output = next((r for r in results if r.get("evaluatorId") == "StrandsEvals.OutputEvaluator"), {})
    called = next((r for r in results if r.get("evaluatorId") == "StrandsEvals.ToolCalled"), {})
    traj = next((r for r in results if r.get("evaluatorId") == "StrandsEvals.TrajectoryEvaluator"), {})
    output_score = output.get("value")
    traj_score = traj.get("value")
    succeeded = (isinstance(output_score, (int, float)) and output_score >= 0.5) or called.get("tool_called")
    if succeeded and isinstance(traj_score, (int, float)) and traj_score < TRAJECTORY_PASS_THRESHOLD:
        flagged = True
        n_calls = called.get("n_tool_calls", "?")
        tokens = local_runs.get(name, {}).get("tokens")
        succ = f"{output_score:.2f}" if isinstance(output_score, (int, float)) else "n/a"
        print(f"\n  [!] {name}: succeeded (TaskSuccess {succ}) in {n_calls} tool call(s) / {tokens} tokens, "
              f"but TrajJudge scored {traj_score:.2f}")
        print(f"      Why: {traj.get('explanation', '')}")
        print("      Trajectory:")
        for i, call in enumerate(local_runs.get(name, {}).get("detailed", []), 1):
            print(f"        {i}. {call}")
if not flagged:
    print("  None flagged: every successful answer was reached via an efficient trajectory.")

### Apply a fix and re-test

The trajectory judge did not just score the baseline low, it named *why*: an unnecessary selector
round-trip, repeated `$metadata` rediscovery, and trial-and-error `$filter` guessing, and it
recommended putting the schema where the query is built so the agent answers in one call. That is a
diagnosis of the **architecture**, not just the prompt: all three wasted steps rediscover, at
runtime, facts that are fixed at design time.

So the fix is architectural. `util/warehouse_agent.py` ships `build_improved_warehouse_agent`, which
adds a deep **`get_warehouse_stock`** tool: the SAP service root, entity set, field names, warehouse
ID, and server-side `$filter`/`$select`/`$orderby` all live *in code*, so a stock question becomes a
single, correct-by-construction call, no selector, no `$metadata`, no fumbled filters. The selector
and generic `odata_caller` stay wired in as a **fallback** for genuinely off-path questions, so the
improved agent is a *router*: deep tool on the hot path, dynamic discovery for everything else. This
maps each judge finding to a concrete code change:

| Judge finding (baseline) | Fix in `get_warehouse_stock` |
|---|---|
| unnecessary selector round-trip | no selector on the stock path; the router calls the tool directly |
| repeated `$metadata` rediscovery | field names compiled into `$select` |
| trial-and-error `$filter` guessing | `$filter` built deterministically in Python |

We re-run the same scenarios against the improved agent and grade it with the **same four Strands
Evals graders** (ToolCalled, TaskSuccess, TrajJudge, ToolSelect), now counting its primary tool,
`get_warehouse_stock`. The next cell prints an overview on the same axes as the baseline, then a
**before → after** table where the trajectory collapses from `odata_caller` × N (with selector +
`$metadata`) to `get_warehouse_stock` × 1. This closes the loop: measure, diagnose, fix, re-measure.


In [ ]:
# Re-run the scenarios against the IMPROVED agent (deep get_warehouse_stock tool + fallback), then
# grade it with the SAME four Strands Evals graders used on the baseline (code-based ToolCalled +
# the three LLM judges), so the fix is measured on every metric, not just the trajectory score.
# We pass IMPROVED_TOOL_NAME so ToolCalled and the expected trajectory target the deep tool the
# improved agent actually uses. Results are stored in `fixed_results`, mirroring `local_results`,
# so the next cell can print a full overview.
print("Re-running scenarios against the improved agent (deep-tool architecture)...\n")

# evaluate_agent runs each scenario once and applies the code-based ToolCalled grader, counting
# get_warehouse_stock (the improved agent's primary tool) instead of odata_caller.
fixed_runs = evaluate_agent(create_improved_warehouse_agent, evaluation_scenarios,
                            tool_name=IMPROVED_TOOL_NAME)

# Grade the fixed runs with the same LLM judges (OutputEvaluator, TrajectoryEvaluator) reused
# from the baseline cell, building per-scenario result lists shaped exactly like `local_results`.
print("\nGrading fixed answers and trajectories with the LLM judges...\n")
fixed_results = {}
for name, run in fixed_runs.items():
    results = [run["tool_called_result"]]  # StrandsEvals.ToolCalled (code-based)

    try:
        out = output_judge.evaluate(run["name_case"])[0]
        results.append({
            "evaluatorId": "StrandsEvals.OutputEvaluator",
            "value": out.score,
            "label": out.label or ("pass" if out.test_pass else "fail"),
            "explanation": out.reason,
        })
    except Exception as e:
        print(f"  {name}: OutputEvaluator ERROR: {e}")

    try:
        out = trajectory_judge.evaluate(run["detail_case"])[0]
        run["traj_score"] = out.score
        run["traj_reason"] = out.reason
        results.append({
            "evaluatorId": "StrandsEvals.TrajectoryEvaluator",
            "value": out.score,
            "label": out.label or ("pass" if out.test_pass else "fail"),
            "explanation": out.reason,
        })
        print(f"  {name}: TaskSuccess + TrajJudge={out.score:.2f}")
    except Exception as e:
        run["traj_score"] = None
        run["traj_reason"] = f"error: {e}"
        print(f"  {name}: TrajectoryEvaluator ERROR: {e}")

    fixed_results[name] = results

# ToolSelectionAccuracy needs OTEL spans, so it runs through the same Experiment + TracedHandler
# harness used on the baseline, here with the improved (deep-tool) agent. Its expected_trajectory
# targets get_warehouse_stock. Guarded and optional.
print("\nGrading fixed tool selection (needs local trace capture)...\n")
try:
    from strands_evals import Experiment, Case, eval_task, TracedHandler

    fixed_cases = [
        Case(name=s["name"], input=s["prompt"], expected_trajectory=[IMPROVED_TOOL_NAME])
        for s in evaluation_scenarios
    ]

    @eval_task(TracedHandler())
    def fixed_warehouse_eval_task():
        return create_improved_warehouse_agent()

    fixed_experiment = Experiment(
        cases=fixed_cases,
        evaluators=[ToolSelectionAccuracyEvaluator(model=model)],
    )
    fixed_reports = await fixed_experiment.run_evaluations_async(fixed_warehouse_eval_task, max_workers=1)

    for report in fixed_reports:
        for case_dict, score in zip(report.cases, report.scores):
            cname = case_dict.get("name") if isinstance(case_dict, dict) else None
            if cname is not None and score is not None:
                fixed_results.setdefault(cname, []).append({
                    "evaluatorId": "StrandsEvals.ToolSelectionAccuracy",
                    "value": score,
                    "label": "justified" if score >= 0.5 else "unjustified",
                    "explanation": f"{report.evaluator_name} tool-selection score.",
                })
                print(f"  {cname}: ToolSelectionAccuracy={score:.2f}")
    print("\nFixed-agent grading complete.")
except Exception as e:
    print(f"\n  ToolSelectionAccuracy skipped for fixed agent (local trace capture unavailable): {e}")
    print("  ToolCalled, TaskSuccess and TrajJudge results above still stand.")

In [ ]:
# Fixed-agent overview + full before/after comparison, across ALL local graders (not just the
# trajectory judge). The first table mirrors the baseline "LOCAL EVALUATION" table above so you
# can read the improved agent on the same axes; the second puts every metric side by side,
# before/after. Note the Calls column: the primary tool changes from odata_caller (baseline) to
# get_warehouse_stock (improved), so this counts N wasteful OData calls collapsing to 1 deep call.


def _score_for(results, eid):
    """Pull one evaluator's score out of a per-scenario result list (or None)."""
    return next((r.get("value") for r in results if r.get("evaluatorId") == eid), None)


def _fmt(score):
    return f"{score:.2f}" if isinstance(score, (int, float)) else "-"


# --- Fixed-agent overview (same columns as the baseline LOCAL EVALUATION table) ---
print("=" * 100)
print(" IMPROVED AGENT (deep get_warehouse_stock tool) - STRANDS EVALS (code-based + LLM-judge)")
print("=" * 100)
header = f"{'Scenario':<25}"
for eid in LOCAL_EVALUATOR_IDS:
    header += f" {_local_short_name(eid):>14}"
header += f" {'Tokens':>10}"
print(header)
print("-" * 100)
for name in (s["name"] for s in evaluation_scenarios):
    row = f"{name:<25}"
    for eid in LOCAL_EVALUATOR_IDS:
        row += f" {_fmt(_score_for(fixed_results.get(name, []), eid)):>14}"
    tokens = fixed_runs.get(name, {}).get("tokens")
    row += f" {tokens:>10}" if isinstance(tokens, int) else f" {'-':>10}"
    print(row)
print("=" * 100)

# --- Before/after, every metric (tool calls, tokens, and all four graders) ---
print("\n" + "=" * 118)
print(" BEFORE -> AFTER (baseline vs. improved agent) - all metrics")
print("=" * 118)
header = f"{'Scenario':<23} {'Calls':>11} {'Tokens':>17}"
for eid in LOCAL_EVALUATOR_IDS:
    header += f" {_local_short_name(eid) + ' b->a':>18}"
print(header)
print("-" * 118)
for name in (s["name"] for s in evaluation_scenarios):
    base, fix = local_runs.get(name, {}), fixed_runs.get(name, {})
    base_calls = base.get("tool_called_result", {}).get("n_tool_calls", "?")
    fix_calls = fix.get("tool_called_result", {}).get("n_tool_calls", "?")
    base_tok, fix_tok = base.get("tokens", "-"), fix.get("tokens", "-")
    row = (f"{name:<23} {f'{base_calls}->{fix_calls}':>11} "
           f"{f'{base_tok}->{fix_tok}':>17}")
    for eid in LOCAL_EVALUATOR_IDS:
        b = _fmt(_score_for(local_results.get(name, []), eid))
        a = _fmt(_score_for(fixed_results.get(name, []), eid))
        row += f" {f'{b}->{a}':>18}"
    print(row)
print("=" * 118)
print("\nReading it: the Calls column tells the story, a multi-call odata_caller trajectory "
      "(selector + $metadata + trial-and-error $filter) collapses to a single get_warehouse_stock "
      "call, so TrajJudge jumps and tokens drop, while TaskSuccess stays high. The deep tool made "
      "the agent faster and more reliable, not just better-prompted. That is the whole point of "
      "grading trajectory alongside outcome, and of fixing the architecture the judge pointed at.")

---

## Stage 2: Deployed evaluation with AgentCore

Stage 1 answered *"is the logic good?"* and produced a fix. But that fix only lives in this
notebook. The deployed runtime still runs the baseline agent. Stage 2 answers the second
question: **did the fix ship, and how does the real thing behave under production infrastructure?**

So the flow here is deploy-then-verify:

1. **Redeploy the improved agent in-place** to the same Lab 7 runtime.
2. **Grade the improved runtime** with **AgentCore Evaluations**, which scores it from its
   **OTEL traces** rather than an in-process result: the deployed agent emits spans to CloudWatch,
   we invoke it per scenario and wait (~180s) for ingestion, then `EvaluationClient.run()` reads
   those spans and scores them with built-in and custom LLM-as-judge evaluators.
3. **Build a scorecard + cross-stage tie-back**: a parity check (Correctness / ToolSelectionAccuracy /
   OpsQuality: did quality hold in prod?) plus prod-only latency, framed as a before/after
   against Stage 1 (local naive to prod improved).

This is what you run against the *actual* production agent, and the same mechanism you would wire
into continuous online evaluation for monitoring.

### Configure evaluation infrastructure

Set up the AgentCore Runtime client and helper functions for invoking the deployed agent and
waiting for OTEL span ingestion.


In [ ]:
# Preflight: CloudWatch Transaction Search must be ON for Stage 2 to have traces to grade.
#
# The deployed agent already emits OTEL spans (the starter toolkit builds the container with
# aws-opentelemetry-distro and runs it under opentelemetry-instrument — no code change needed).
# But AgentCore Evaluations reads those spans as X-Ray *traces*, and spans are only indexed into
# traces once Transaction Search is enabled at the account level. If it's off, EvaluationClient.run()
# below finds nothing to grade and every built-in score comes back empty — a confusing failure to
# debug after a 4-min redeploy and a 180s ingestion wait. So we check it up front with a read-only
# X-Ray call and fail loud with setup instructions, rather than let Stage 2 silently grade nothing.
xray_client = boto3.client("xray", region_name=REGION)

try:
    dest = xray_client.get_trace_segment_destination()
    destination = dest.get("Destination")
    status = dest.get("Status")
    if destination == "CloudWatchLogs" and status == "ACTIVE":
        print(f"Transaction Search is enabled (destination={destination}, status={status}).")
        print("Stage 2 will be able to read the deployed agent's traces.")
    else:
        raise SystemExit(
            "CloudWatch Transaction Search is not fully enabled "
            f"(destination={destination}, status={status}). Stage 2 needs it to read the "
            "deployed agent's OTEL spans as traces.\n\n"
            "Enable it once per account/region:\n"
            "  CloudWatch console -> Application Signals -> Transaction Search -> Enable,\n"
            "  or: aws xray update-trace-segment-destination --destination CloudWatchLogs\n"
            "     aws xray update-indexing-rule ...  (see the Transaction Search setup docs)\n\n"
            "Docs: https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/observability-configure.html"
        )
except SystemExit:
    raise
except Exception as e:
    # A permissions or API error here shouldn't hard-block Stage 2 (the account may still be
    # configured); surface it as a warning so the user can decide whether to proceed.
    print(f"WARNING: could not verify Transaction Search status ({e}).")
    print("If Stage 2 grading returns empty scores, confirm Transaction Search is enabled for this account/region.")

In [ ]:
# AgentCore Runtime client for invoking the deployed agent
agentcore_client = boto3.client("bedrock-agentcore", region_name=REGION)

# CloudWatch log group where OTEL spans land
CW_LOG_GROUP = f"/aws/bedrock-agentcore/runtimes/{AGENT_ID}-DEFAULT"

# Time to wait for OTEL spans to arrive in CloudWatch.
INGESTION_DELAY = 180


def invoke_with_retry(client, agent_arn, session_id, prompt, max_retries=3, wait=30):
    """Invoke the deployed agent, retrying cold-start 500s under a FRESH session id each time.

    Returns (response, effective_session_id, cold_start). A single retry loop owns every
    invocation, so exactly one id per attempt is ever sent to the runtime: the first attempt
    uses `session_id`, and each retry appends a `_r{n}` suffix BEFORE re-invoking. The returned
    `effective_session_id` is therefore the id the successful attempt ran under, i.e. the id
    whose OTEL spans exist, which is exactly what grade_sessions must query.
    `cold_start` is True if any retry was needed. Raises the last error if all attempts fail.
    """
    effective_session_id = session_id
    for attempt in range(max_retries):
        try:
            response = client.invoke_agent_runtime(
                agentRuntimeArn=agent_arn,
                qualifier="DEFAULT",
                runtimeSessionId=effective_session_id,
                payload=json.dumps({"prompt": prompt}).encode("utf-8"),
            )
            return response, effective_session_id, attempt > 0
        except client.exceptions.RuntimeClientError as e:
            if attempt < max_retries - 1:
                print(f"  Runtime error (attempt {attempt + 1}/{max_retries}), retrying in {wait}s (likely cold start)...")
                time.sleep(wait)
                # Mutate BEFORE the next invoke so the retried call and its spans share one id.
                effective_session_id = f"{session_id}_r{attempt + 1}"
            else:
                raise e
    return None, effective_session_id, True


def invoke_with_timing(client, agent_arn, session_id, prompt):
    """Wrap invoke_with_retry to record wall-clock latency and the cold-start flag.

    Returns (response, timing) where timing["session_id"] is the EFFECTIVE id the invocation
    succeeded under (see invoke_with_retry), so downstream span lookups query the id whose
    spans actually exist.
    """
    start = time.time()
    response, effective_session_id, cold = invoke_with_retry(client, agent_arn, session_id, prompt)
    return response, {
        "latency_s": time.time() - start,
        "cold_start": cold,
        "session_id": effective_session_id,
    }


print(f"Agent ID: {AGENT_ID}")
print(f"Agent ARN: {AGENT_ARN}")
print(f"CloudWatch Log Group: {CW_LOG_GROUP}")
print(f"Ingestion delay: {INGESTION_DELAY}s")

### Create the custom evaluator

We register a domain-specific LLM-as-a-judge evaluator in the AgentCore control plane. This complements the built-in evaluators with SAP warehouse-specific scoring criteria that generic evaluators cannot assess.

**WarehouseOperationalQuality** (TRACE-level): evaluates whether the agent's response is operationally useful for a warehouse manager: does it provide actionable inventory insights, use correct product codes, and present data in a way that supports procurement decisions?

In [ ]:
agentcore_control = boto3.client("bedrock-agentcore-control", region_name=REGION)

_SUFFIX = uuid.uuid4().hex[:8]

# TODO: Use the inference profile matching your region (us.* for us-east-1, eu.* for eu-central-1)
JUDGE_MODEL_ID = "us.amazon.nova-pro-v1:0"

# Custom TRACE-level evaluator: Warehouse Operational Quality
# NOTE: `instructions` carries only the rubric (what to judge and when each score applies). We
# deliberately do NOT hand-write output-format directives ("respond with the score on the first
# line", etc.): the service derives the response format from the `ratingScale` below and appends
# its own formatting scaffolding to the prompt. A hand-written format block would duplicate — and
# risk contradicting — that, which can make the judge's output unparseable and silently drop scores.
print("Creating WarehouseOperationalQuality evaluator (TRACE-level)...")
warehouse_quality_response = agentcore_control.create_evaluator(
    evaluatorName=f"WarehouseOperationalQuality_{_SUFFIX}",
    level="TRACE",
    evaluatorConfig={
        "llmAsAJudge": {
            "instructions": (
                "You are a warehouse operations expert evaluating an AI assistant that queries "
                "SAP S/4HANA warehouse APIs for inventory management.\n\n"
                "Conversation context: {context}\n"
                "Agent response: {assistant_turn}\n"
                "Expected behavior: {expected_response}\n\n"
                "Evaluate the OPERATIONAL QUALITY of the response for a warehouse manager. Score based on:\n"
                "1. Does the response contain specific, quantitative inventory data (not vague statements)?\n"
                "2. Are SAP product codes (WM-AN01, WM-AN02, WM-AN03, WM-AN04) used correctly?\n"
                "3. Is the data presented in a way that supports immediate operational decisions "
                "(e.g., reorder recommendations, fulfillment feasibility, capacity utilization)?\n"
                "4. Does the response avoid hallucinating inventory numbers when API data is unavailable?\n\n"
                "Important: If the agent successfully queried the API and returned real data with "
                "correct product codes and actionable insights, score 1.0 even if formatting differs "
                "from the expected response."
            ),
            "ratingScale": {
                "numerical": [
                    {"value": 0.0, "label": "not_actionable", "definition": "Response lacks data, hallucinates numbers, or provides no operational value."},
                    {"value": 0.5, "label": "partially_useful", "definition": "Some useful data present but missing key operational context for decisions."},
                    {"value": 1.0, "label": "operationally_excellent", "definition": "Accurate, specific, and actionable warehouse intelligence."},
                ]
            },
            "modelConfig": {
                "bedrockEvaluatorModelConfig": {
                    "modelId": JUDGE_MODEL_ID,
                    "inferenceConfig": {"maxTokens": 512},
                }
            },
        }
    },
)
CUSTOM_EVALUATOR_ID = warehouse_quality_response["evaluatorId"]
print(f"  Created: {CUSTOM_EVALUATOR_ID}")
print(f"\nCustom evaluator registered in AgentCore control plane.")

### Ship the Stage-1 fix: redeploy the improved runtime

Stage 1 found the baseline agent correct-but-inefficient and produced a fix, the deep-tool
architecture (`build_improved_warehouse_agent`), but that fix only lives in this notebook. The
deployed runtime still runs the baseline agent. Before we can verify the fix in production, we have
to ship it. The next cells redeploy the improved agent **in-place** to the same Lab 7 runtime
(reusing `AGENT_NAME` with `auto_update_on_conflict=True`, so `AGENT_ID` / `AGENT_ARN` stay
valid). The agent is deployed without memory: Lab 9 grades query logic and efficiency, not
personalization, so this keeps the runtime lean and independent of Lab 8.

> **Time and cost:** the redeploy cell triggers a real CodeBuild and takes about 4 min; the full
> Stage 2 (redeploy, invoke per scenario, ~180s OTEL ingestion wait, grade) runs about 9-12 min
> total. It also makes live Bedrock/AgentCore calls, so it incurs cost.

In [ ]:
# Redeploy the improved agent to the SAME AgentCore Runtime from Lab 7. deploy_improved_agent
# (util/deploy_agentcore.py) writes the entrypoint + requirements, patches the Dockerfile, launches
# in place (auto_update_on_conflict, so AGENT_ID / AGENT_ARN stay valid), and waits for READY,
# raising if the redeploy fails so we never grade a stale (baseline) runtime. The deployed
# entrypoint imports the same build_improved_warehouse_agent graded locally in Stage 1, so prod
# runs the exact deep-tool architecture we fixed. Deployment mechanics live in util/ to keep this
# an evaluation lab; open that file if you want to see the AgentCore plumbing.
from util.deploy_agentcore import deploy_improved_agent

deploy_improved_agent(
    agent_name=AGENT_NAME,
    region=REGION,
    sap_api_key=os.environ["SAP_S4HANA_PUBLIC_CLOUD_KEY"],
    model_id=model.get_config()["model_id"],
)
print(f"The improved agent (AGENT_ID={AGENT_ID}) is now deployed and ready for evaluation.")

### Invoke the deployed agent for each scenario

We invoke the redeployed (improved) agent for each evaluation scenario. Each invocation gets a unique `runtimeSessionId` so the evaluator can locate its spans independently.

In [ ]:
# Invoke the deployed agent for each scenario with timing, and collect per-scenario sessions.


def parse_agent_response(response_body: str) -> str:
    """Parse invoke_agent_runtime response into plain text.

    The response is a JSON object: {"role": "assistant", "content": [{"text": "..."}], "metadata": {...}}
    It may arrive as a single blob or as multiple newline-delimited chunks that concatenate into one object.
    """
    text_parts = []

    # Try parsing as a single JSON object (most common)
    try:
        data = json.loads(response_body)
        if isinstance(data, dict) and "content" in data:
            for block in data["content"]:
                if isinstance(block, dict) and "text" in block:
                    text_parts.append(block["text"])
            return "".join(text_parts)
    except json.JSONDecodeError:
        pass

    # Fall back: response may be multiple concatenated JSON chunks (chunked transfer)
    try:
        combined = "".join(response_body.strip().split("\n"))
        data = json.loads(combined)
        if isinstance(data, dict) and "content" in data:
            for block in data["content"]:
                if isinstance(block, dict) and "text" in block:
                    text_parts.append(block["text"])
            return "".join(text_parts)
    except json.JSONDecodeError:
        pass

    # Last resort: return raw truncated
    return response_body[:500]


def run_prod_round(label, scenarios):
    """Invoke each scenario against the deployed runtime with timing, wait for
    span ingestion, and return per-scenario session dicts (timing included)."""
    sessions = []
    print(f"[{label}] Invoking deployed agent for {len(scenarios)} scenarios...\n")
    for scenario in scenarios:
        session_id = f"eval_{scenario['name']}_{uuid.uuid4().hex}"  # >= 33 chars
        try:
            response, timing = invoke_with_timing(agentcore_client, AGENT_ARN, session_id, scenario["prompt"])
            response_body = response["response"].read().decode("utf-8")
            agent_text = parse_agent_response(response_body)
            # Store the EFFECTIVE session id the invocation succeeded under (timing["session_id"]),
            # not the pre-generated one: a cold-start retry mutates the id, and grade_sessions
            # must query the id whose spans actually exist.
            sessions.append({
                "scenario_name": scenario["name"], "session_id": timing["session_id"],
                "prompt": scenario["prompt"], "response": agent_text[:500],
                "expected_response": scenario["expected_response"],
                "expected_trajectory": scenario["expected_trajectory"],
                "assertions": scenario["assertions"],
                "latency_s": timing["latency_s"], "cold_start": timing["cold_start"],
            })
            print(f"  [{scenario['name']}] {timing['latency_s']:.1f}s"
                  f"{' (cold start)' if timing['cold_start'] else ''}")
        except Exception as e:
            print(f"  [{scenario['name']}] ERROR: {e}")
    print(f"\n[{label}] Waiting {INGESTION_DELAY}s for CloudWatch span ingestion...")
    time.sleep(INGESTION_DELAY)
    return sessions

### Evaluate with built-in evaluators

We use `EvaluationClient.run()` to score each session with AgentCore's built-in evaluators:

| Evaluator | Level | Needs Ground Truth | What it measures |
|-----------|-------|-------------------|-----------------|
| `Builtin.Correctness` | TRACE | `expected_response` | Factual accuracy of the response |
| `Builtin.Helpfulness` | TRACE | None | How useful/valuable the response is |
| `Builtin.GoalSuccessRate` | SESSION | `assertions` | Whether the agent completed the user's goal |
| `Builtin.ToolSelectionAccuracy` | TOOL_CALL | None | Whether each individual tool call was justified |

In [ ]:
from bedrock_agentcore.evaluation import EvaluationClient, ReferenceInputs

ec = EvaluationClient(region_name=REGION)

# Pre-populate the evaluator-level cache. This is the documented setup for Builtin.* evaluators:
# EvaluationClient.run() resolves each evaluator's level (SESSION / TRACE / TOOL_CALL) to build the
# right evaluationTarget, but its get_evaluator lookup does NOT return a level for built-in IDs, so
# on that path the SDK silently falls back to SESSION — wrong for TRACE and TOOL_CALL evaluators.
# Pre-seeding the cache is what prevents that mis-classification, and it's exactly what the official
# aws-samples example does (amazon-bedrock-agentcore-samples,
# 01-features/06-observe-evaluate-optimize-your-agent). Levels are from the built-in prompt-templates
# docs: https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/prompt-templates-builtin.html
# NOTE: ToolSelectionAccuracy is a TOOL_CALL (tool-level) evaluator, not SESSION — the docs classify
# it under "Tool-level evaluators" (it judges whether each individual tool call was justified).
ec._evaluator_level_cache.update({
    "Builtin.Correctness": "TRACE",
    "Builtin.Helpfulness": "TRACE",
    "Builtin.GoalSuccessRate": "SESSION",
    "Builtin.ToolSelectionAccuracy": "TOOL_CALL",
})

BUILTIN_EVALUATOR_IDS = [
    "Builtin.Correctness",
    "Builtin.Helpfulness",
    "Builtin.GoalSuccessRate",
    "Builtin.ToolSelectionAccuracy",
]

# The custom WarehouseOperationalQuality evaluator (registered above) is TRACE-level too, so
# grade_sessions can run it in the same pass as the built-ins.
ec._evaluator_level_cache[CUSTOM_EVALUATOR_ID] = "TRACE"


def grade_sessions(sessions):
    """Grade prod sessions with the built-in + custom AgentCore evaluators.

    Runs BUILTIN_EVALUATOR_IDS then the custom WarehouseOperationalQuality evaluator on each
    session's OTEL traces and returns {scenario_name: [built-in results..., custom results...]}
    the combined shape the scorecard and display consume. Any evaluator error falls back to
    [] for that group so one bad session never aborts the round.
    """
    graded = {}
    print(f"Grading {len(sessions)} session(s) with AgentCore built-in + custom evaluators...\n")
    for session in sessions:
        name = session["scenario_name"]
        print(f"  Evaluating: {name} (session: {session['session_id']})")

        reference_inputs = ReferenceInputs(
            expected_response=session["expected_response"],
            expected_trajectory=session["expected_trajectory"],
            assertions=[session["assertions"]],
        )

        try:
            builtin = ec.run(
                evaluator_ids=BUILTIN_EVALUATOR_IDS,
                agent_id=AGENT_ID,
                session_id=session["session_id"],
                look_back_time=timedelta(hours=1),
                reference_inputs=reference_inputs,
            )
            for r in builtin:
                print(f"    {r.get('evaluatorId', 'unknown')}: {r.get('value', 'N/A')} ({r.get('label', '')})")
        except Exception as e:
            print(f"    built-in ERROR: {e}")
            builtin = []

        try:
            custom = ec.run(
                evaluator_ids=[CUSTOM_EVALUATOR_ID],
                agent_id=AGENT_ID,
                session_id=session["session_id"],
                look_back_time=timedelta(hours=1),
                reference_inputs=reference_inputs,
            )
            for r in custom:
                label = r.get("label", "")
                print(f"    OpsQuality: {r.get('value', 'N/A')} ({label})")
                if r.get("explanation"):
                    print(f"      {r['explanation'][:120]}")
        except Exception as e:
            print(f"    custom ERROR: {e}")
            custom = []

        graded[name] = builtin + custom
        print()

    print("Grading complete.")
    return graded


### Aggregate one round into a scorecard

`grade_sessions` (above) runs the built-in **and** the custom **WarehouseOperationalQuality**
evaluator on every session. The custom judge scores whether the agent provides actionable
warehouse intelligence, something the generic built-ins cannot assess. The next cell adds
`build_scorecard`, which folds one graded round into a flat metric dict (Correctness,
ToolSelectionAccuracy, OpsQuality, plus prod-only latency) ready for `render_scorecard`.


In [ ]:
# Aggregate one graded prod round into a flat scorecard dict for render_scorecard.
from util.deployed_eval import summarize_timing


def build_scorecard(sessions, graded):
    """Aggregate one prod round into a flat metric dict for render_scorecard."""
    def avg_metric(eid):
        vals = [
            r.get("value") for s in sessions
            for r in graded.get(s["scenario_name"], [])
            if r.get("evaluatorId") == eid and isinstance(r.get("value"), (int, float))
        ]
        return sum(vals) / len(vals) if vals else None

    timing = summarize_timing([{"latency_s": s["latency_s"], "cold_start": s["cold_start"]} for s in sessions])
    return {
        "Correctness": avg_metric("Builtin.Correctness"),
        "ToolSelectionAccuracy": avg_metric("Builtin.ToolSelectionAccuracy"),
        "OpsQuality": avg_metric(CUSTOM_EVALUATOR_ID),
        "latency_s": timing["mean_s"],
    }


## Grade the improved runtime and render the scorecard

Now we run the whole prod round end to end: `run_prod_round` invokes the deployed (improved,
redeployed) agent for every scenario, `grade_sessions` scores each one, and `build_scorecard`
folds the round into a single scorecard.

Read the scorecard as a parity check plus one prod-only signal. Correctness, ToolSelectionAccuracy
and OpsQuality confirm the Stage-1 fix **did not break quality**: it holds in production, just as it
did locally. Latency is what a local run can't produce: it is the price of the real runtime, and the
payoff of the deep-tool fix (fewer tool calls, fewer tokens) measured where it actually runs. The
before→after narrative is therefore **Stage-1 local (naive baseline, flagged inefficient) →
Stage-2 prod (improved deep-tool agent, measured on the live runtime)**.

> `run_prod_round` does a live AgentCore invocation per scenario and then waits ~180s for OTEL
> spans to land in CloudWatch, so this cell is slow. Run it once.


In [ ]:
from util.deployed_eval import render_scorecard

# Run the full prod round: invoke the deployed (improved) runtime, grade every session, and
# fold the round into one scorecard. run_prod_round does live invocations + a ~180s ingestion
# wait, so this is the slow cell, run it once.
improved_sessions = run_prod_round("improved", evaluation_scenarios)
improved_graded = grade_sessions(improved_sessions)
improved_scorecard = build_scorecard(improved_sessions, improved_graded)

METRIC_KEYS = ["Correctness", "ToolSelectionAccuracy", "OpsQuality", "latency_s"]

print("=" * 42)
print(" STAGE 2: IMPROVED RUNTIME (prod, redeployed)")
print("=" * 42)
print(render_scorecard(improved_scorecard, METRIC_KEYS))
print()

# Cross-stage before->after: Stage-1 LOCAL graded the naive prompt and its trajectory judge
# flagged the low-stock scenario as inefficient; Stage-2 PROD grades the improved prompt on
# the real runtime. The latency above is what that fix buys, measured in production.
for scenario_name in local_results:
    local_traj = next(
        (r for r in local_results.get(scenario_name, [])
         if r.get("evaluatorId") == "StrandsEvals.TrajectoryEvaluator"),
        {},
    )
    traj_score = local_traj.get("value")
    if isinstance(traj_score, (int, float)) and traj_score < 0.5:
        # latency can be None if the whole round errored (build_scorecard returns
        # latency_s=None), so guard before formatting.
        lat = improved_scorecard["latency_s"]
        lat_str = f"{lat:.1f}s" if isinstance(lat, (int, float)) else "n/a"
        print(f"Cross-stage: Stage-1 local flagged '{scenario_name}' as inefficient "
              f"(TrajJudge {traj_score:.2f}). Stage-2 prod ran the fixed prompt above, "
              f"mean latency {lat_str}.")

## Cleanup (optional)

In [ ]:
# Uncomment to clean up the custom evaluator created in this lab.
# The deployed runtime is owned by Lab 7; clean it up from that notebook.

# agentcore_control.delete_evaluator(evaluatorId=CUSTOM_EVALUATOR_ID)
# print(f"Deleted evaluator: {CUSTOM_EVALUATOR_ID}")

## Summary

You evaluated the warehouse agent in **two stages**, grounded in Anthropic's [Demystifying evals for AI agents](https://www.anthropic.com/engineering/demystifying-evals-for-ai-agents).

**Grader taxonomy:** code-based (fast, objective), model-based / LLM-as-judge (flexible, nuanced),
human (gold standard, for calibration), plus the outcome-vs-trajectory distinction.

**Stage 1: Local evaluation with Strands Evals** (fast, cheap pre-deploy gate):
- Code-based `ToolCalled`; LLM-as-judge `TrajectoryEvaluator` (fed each call's OData arguments so it
  *names* the wasteful calls) and `ToolSelectionAccuracyEvaluator`, plus token counts as an
  informational efficiency signal, all routed through the same `SAPGenAIHubModel`.
- Caught a **correct-but-inefficient** answer (the low-stock query) that outcome graders score 1.0,
  then **applied the judge's fix and re-tested**. The judge's diagnosis pointed past the prompt to
  the *architecture*: it was rediscovering, at runtime, facts fixed at design time. So the fix is a
  deep **`get_warehouse_stock`** tool that encodes the schema and query in code (with the selector +
  `odata_caller` kept as a fallback), collapsing an N-call `odata_caller` trajectory to a single
  call while keeping the answer correct, the full measure → diagnose → fix → re-measure loop.

**Stage 2: Deployed evaluation with AgentCore** (deploy-then-verify on the real runtime):
- **Shipped the Stage-1 fix**: redeployed the improved deep-tool agent in-place to the live AgentCore
  Runtime, because the fix had only ever lived locally.
- **Verified it in production** from OTEL traces: built-in evaluators (Correctness, Helpfulness,
  GoalSuccessRate, ToolSelectionAccuracy) plus a custom LLM-as-judge evaluator
  (WarehouseOperationalQuality) confirmed quality **held (parity)**, alongside **prod-only latency**,
  the signal a local run physically can't produce.
- The before→after is therefore **cross-stage**: Stage-1 local (naive baseline, flagged inefficient)
  → Stage-2 prod (improved deep-tool agent, measured where it actually runs), no separate prod baseline round.

**Key takeaway:** evaluate locally first, then ship the fix and verify it in production; combine
code-based and model-based graders so you measure not just *whether* the agent is right (outcome)
but *how efficiently* it gets there (trajectory); and when the trajectory judge keeps naming the
same runtime rediscovery, fix the **architecture** it points at (a deep, typed tool), not just the
prompt, then re-grade.

**Next steps (see the evaluation harness spec):** negative / out-of-scope scenarios, multi-trial
runs with pass@k / pass^k, chaos/fault-injection (Strands Evals 1.0+), and a pass/fail regression gate.